# Topic: SQL Percentile Calculation Pattern

## Definition (30-second explanation)
* A percentile tells you what percentage of values fall below a given value in a dataset.
* For example, if a student's score is at the 90th percentile, 90% of all students scored below them.
* SQL provides specialized window functions like `PERCENT_RANK()`, `CUME_DIST()`, `NTILE()`, and `PERCENTILE_CONT()` to compute these metrics efficiently without self-joins.

## Why Interviewers Ask This
* To test your ability to perform relative rankings and statistical bucketizations (like deciles and quartiles), which are foundational for Data Science tasks.
* To evaluate if you understand the subtle mathematical differences between cumulative distribution and relative positioning.
* To check if you know the specific syntax required for ordered-set aggregate functions (like median calculations).

## Core Concepts
* **PERCENT_RANK():** Calculates the relative position of a row, always ranging from 0.0 to 1.0 (the first row is always 0).
* **CUME_DIST():** Calculates the cumulative distribution, representing the fraction of rows less than or equal to the current row (never 0).
* **NTILE(n):** Divides the result set into `n` equal buckets, returning the bucket number (1 to n) for each row.
* **PERCENTILE_CONT(p) vs PERCENTILE_DISC(p):** `CONT` interpolates a continuous value that might not exist in the dataset (like an exact median), whereas `DISC` returns an actual discrete row value.

## When to Use
* **NTILE:** Use for cohort analysis or segmenting data into quartiles/deciles (e.g., categorizing top 25% of customers).
* **PERCENT_RANK:** Use for benchmarking a specific entity's relative position (e.g., where an employee's salary sits compared to peers).
* **PERCENTILE_CONT:** Use for finding precise statistical thresholds, most commonly the median (50th percentile) or interquartile range (IQR).
* **CUME_DIST:** Use when you need to know the total proportion of a population that falls at or below a certain threshold (e.g., Value at Risk in finance).

## Advantages
* Offloads heavy statistical calculations from Python/Pandas directly into the database engine, reducing data transfer size.
* Handles ties gracefully and mathematically correctly based on standard statistical formulas.

## Limitations
* `PERCENTILE_CONT` and `PERCENTILE_DISC` syntax (`WITHIN GROUP`) varies slightly across SQL dialects (e.g., PostgreSQL vs MySQL), and some older systems lack them entirely.
* Extreme outliers can skew `PERCENTILE_CONT` interpolations in very small datasets.

## Common Comparisons
* **PERCENT_RANK vs CUME_DIST for Ties:** If two rows tie, `PERCENT_RANK` gives them the same relative rank based on the start of the tie, whereas `CUME_DIST` calculates based on the maximum position of the tied rows.
* **PERCENTILE_CONT vs PERCENTILE_DISC:** `CONT` can generate a new number (e.g., averaging the two middle numbers for an even-count median), while `DISC` will pick one of the actual existing middle numbers.

## Common Interview Traps
* **Confusing PERCENT_RANK with CUME_DIST:** Remembering that `PERCENT_RANK` starts at 0 and `CUME_DIST` never hits 0 is a frequent interview follow-up.
* **Wrong ORDER BY direction:** `ORDER BY ASC` gives the fraction that is *less* than the current value; `DESC` gives the fraction that is *more*.
* **PERCENTILE_CONT Syntax:** Applying `OVER(PARTITION BY...)` to `PERCENTILE_CONT` instead of the required `WITHIN GROUP (ORDER BY ...)` syntax.
* **Forgetting to multiply by 100:** Both `PERCENT_RANK` and `CUME_DIST` return a decimal from 0.0 to 1.0; you often need to `* 100` and `ROUND()` for standard business readability.

## SQL Syntax 
```sql
-- Syntax for standard window functions
SELECT 
    PERCENT_RANK() OVER (PARTITION BY dept ORDER BY salary ASC) as pr,
    NTILE(4) OVER (ORDER BY salary DESC) as quartile
FROM employees;

-- Syntax for ordered-set aggregate functions (e.g., PostgreSQL/SQL Server)
SELECT 
    dept,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary ASC) as median_salary
FROM employees
GROUP BY dept;
```

## Important Formula
* `PERCENT_RANK()` = `(rank - 1) / (total_rows - 1)`.
* `CUME_DIST()` = `rank / total_rows`.

## 45-Second Interview Answer
"For percentile calculations in SQL, I rely on four main functions depending on the business need. I use NTILE() for simple bucketing like deciles or quartiles. If I need a relative percentile score for individual rows, I use PERCENT_RANK() or CUME_DIST(), being careful to multiply by 100. If I need to extract a specific statistical threshold from an aggregated group, like finding the exact median salary, I use PERCENTILE_CONT() combined with the WITHIN GROUP clause."

## Resume / Project Connection 
* **HR Analytics:** Salary benchmarking to determine which percentile each employee's salary falls into.
* **Marketing:** Customer Lifetime Value (LTV) percentile segmentation.
* **Finance:** Calculating Value at Risk (VaR) by identifying the 95th percentile of daily losses.